In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        data_root = candidate / 'Step0_Data' / '1_Data'
        if (data_root / 'WDI' / 'WDIEXCEL-2026_02_01.xlsx').is_file():
            return candidate
    raise FileNotFoundError('Could not locate standalone release root from the current working directory')

release_root = find_release_root(cwd)
shared_data_root = release_root / 'Step0_Data' / '1_Data'
step_root = release_root / 'Step1_FeaturePool'
wdi_dir = shared_data_root / 'WDI'
wdi_path = wdi_dir / 'WDIEXCEL-2026_02_01.xlsx'
scope_path = wdi_dir / 'WDI_182countries_1990_2024_2017PPP.xlsx'
output_path = step_root / '2_Results' / '0_wdi_all_indicators_coverage.csv'
years = [str(y) for y in range(1990, 2025)]
possible_iso_cols = ['ISO3', 'ISO3 Code', 'Country Code', 'Code']

In [ ]:
scope_df = pd.read_excel(scope_path, sheet_name='GDP_PPP_2017')
iso_col = next((col for col in possible_iso_cols if col in scope_df.columns), scope_df.columns[0])
target_iso_list = scope_df[iso_col].dropna().astype(str).unique().tolist()
if len(target_iso_list) != 182:
    raise ValueError(f'Target country scope must be 182, got {len(target_iso_list)}')
len(target_iso_list)

In [ ]:
wdi_full = pd.read_excel(wdi_path, sheet_name='Data')
column_lookup = {str(col): col for col in wdi_full.columns}
missing_years = [year for year in years if year not in column_lookup]
if missing_years:
    raise ValueError(f'WDI source missing year columns: {missing_years}')
year_cols = [column_lookup[year] for year in years]
wdi_filtered = wdi_full[wdi_full['Country Code'].astype(str).isin(target_iso_list)].copy()
wdi_filtered.shape

In [ ]:
wdi_long = pd.melt(
    wdi_filtered,
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Value',
)
coverage = wdi_long.groupby(['Indicator Code', 'Indicator Name'])['Value'].count().reset_index()
coverage.columns = ['Indicator Code', 'Indicator Name', 'Valid_Count']
coverage['Coverage_Rate'] = coverage['Valid_Count'] / (len(target_iso_list) * len(years))
coverage = coverage.sort_values('Coverage_Rate', ascending=False).reset_index(drop=True)
output_path.parent.mkdir(parents=True, exist_ok=True)
coverage.to_csv(output_path, index=False)
coverage.head(10)